# Two dots do not make a curve

Look at a real function only at $0$ and at $1$. The zero function and the hump $t(1-t)$ both give the sample pair $(0,0)$. At the midpoint the first is $0$ and the second is $1/4$. Two dots are two dots. They do not determine the height in the middle.

This notebook uses the shipped calculator. Arithmetic is exact (fractions, not floating point).

Run from the package root, or from this `jupyter_notebooks/` folder.

In [1]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
root = cwd.parent if cwd.name == "jupyter_notebooks" else cwd
code_dir = (root / "code").resolve()
assert (code_dir / "coda_cs_q_bridge_core.py").is_file(), "Run from this package root or jupyter_notebooks/."
sys.path[:] = [str(code_dir)] + [entry for entry in sys.path if entry != str(code_dir)]
import coda_cs_q_bridge_core as companion
assert Path(companion.__file__).resolve() == code_dir / "coda_cs_q_bridge_core.py", "An ambient module shadowed the local companion."
import cochain_presentation as cochain
assert Path(cochain.__file__).resolve() == code_dir / "cochain_presentation.py", "An ambient solver shadowed the local solver."

from sympy import Rational
from coda_cs_q_bridge_core import (
    CofinalShadowTower,
    ProfileFactorizationSolver,
    QueryProbe,
    ShadowNode,
)

## Two functions, the same two dots

Write each function as the three values you need: the ends, and the midpoint you would like to read. The profile keeps only the ends.

In [2]:
zero = Rational(0)
one = Rational(1)
half = Rational(1, 2)

f_flat = {zero: zero, one: zero, half: zero}
f_hump = {zero: zero, one: zero, half: half * (one - half)}

print("flat at 0, 1, 1/2:", f_flat[zero], f_flat[one], f_flat[half])
print("hump at 0, 1, 1/2:", f_hump[zero], f_hump[one], f_hump[half])
print("same two dots?", (f_flat[zero], f_flat[one]) == (f_hump[zero], f_hump[one]))
print("same midpoint?", f_flat[half] == f_hump[half])
assert f_hump[half] == Rational(1, 4)
assert (f_flat[zero], f_flat[one]) == (f_hump[zero], f_hump[one])
assert f_flat[half] != f_hump[half]

flat at 0, 1, 1/2: 0 0 0
hump at 0, 1, 1/2: 0 0 1/4
same two dots? True
same midpoint? False


## The calculator refuses a midpoint factor

Theorem CSQ-T4: the midpoint query does not factor through the two-dot profile. The calculator sees the fibre collision and does not build an image map.

In [3]:
nodes = [ShadowNode("ends", 2)]
projectors = {"ends": lambda f: (f[zero], f[one])}
tower = CofinalShadowTower(nodes, projectors, {})
domain = [f_flat, f_hump]
query = QueryProbe("midpoint", degree=1, eval_fn=lambda f: f[half])
ok, q_bar, err = ProfileFactorizationSolver(tower, query, domain).solve_factorization()
print("factor built?", ok)
print("reason:", err)
assert ok is False
assert q_bar is None
assert "Fiber constancy violation" in str(err)

factor built? False
reason: Fiber constancy violation on fiber ((0, 0),): q({0: 0, 1: 0, 1/2: 0}) = 0 != q({0: 0, 1: 0, 1/2: 1/4}) = 1/4
